## Метод потенциалов

In [20]:
import numpy as np
from IPython.display import display, HTML

def display_transportation_table(costs, basis_matrix, supply, demand, title=""):
    m, n = costs.shape
    
    html = f"<h3 style='color: #e0e0e0; font-family: sans-serif; margin-top: 20px;'>{title}</h3>"
    html += "<table style='border-collapse: collapse; text-align: center; font-family: sans-serif; min-width: 500px; background-color: #1e1e1e; color: #e0e0e0; border: 1px solid #444;'>"
    
    html += "<tr style='background-color: #2d2d2d; border: 1px solid #444;'>"
    html += "<th style='border: 1px solid #444; padding: 10px; color: #ffffff;'>Пункты</th>"
    for j in range(n):
        html += f"<th style='border: 1px solid #444; padding: 10px; color: #ffffff;'>B<sub>{j+1}</sub></th>"
    html += "<th style='border: 1px solid #444; padding: 10px; color: #ffffff;'>Запасы</th>"
    html += "</tr>"
    
    for i in range(m):
        html += "<tr>"
        html += f"<td style='border: 1px solid #444; font-weight: bold; background-color: #2d2d2d; color: #ffffff; padding: 10px;'>A<sub>{i+1}</sub></td>"
        
        for j in range(n):
            val = basis_matrix[i, j]
            if val is None:
                display_val = ""
            elif val == 0:
                display_val = "•"
            else:
                display_val = f"{int(val)}"
                
            cell_html = (
                f"<td style='border: 1px solid #444; width: 90px; height: 55px; position: relative; padding: 2px; background-color: #1e1e1e;'>"
                f"<div style='position: absolute; top: 3px; left: 6px; font-size: 11px; color: #888888;'>{int(costs[i,j])}</div>"
                f"<div style='position: absolute; bottom: 3px; right: 10px; font-weight: bold; font-size: 16px; color: #ffffff;'>{display_val}</div>"
                f"</td>"
            )
            html += cell_html
            
        html += f"<td style='border: 1px solid #444; font-weight: bold; background-color: #252525;'>{int(supply[i])}</td>"
        html += "</tr>"
        
    html += "<tr style='background-color: #2d2d2d; font-weight: bold;'>"
    html += "<td style='border: 1px solid #444; padding: 10px; color: #ffffff;'>Потребности</td>"
    for j in range(n):
        html += f"<td style='border: 1px solid #444;'>{int(demand[j])}</td>"
    html += f"<td style='border: 1px solid #444; background-color: #3d3d3d; color: #ffffff;'>{int(sum(supply))}</td>"
    html += "</tr>"
    html += "</table>"
    
    display(HTML(html))

def solve_transportation_universal(costs, supply, demand):
    costs = np.array(costs, dtype=float)
    supply = np.array(supply, dtype=float)
    demand = np.array(demand, dtype=float)
    
    orig_m, orig_n = costs.shape
    sum_supply = sum(supply)
    sum_demand = sum(demand)
    
    added_row = False
    added_col = False
    
    if sum_supply > sum_demand:
        diff = sum_supply - sum_demand
        costs = np.hstack((costs, np.zeros((orig_m, 1))))
        demand = np.append(demand, diff)
        added_col = True
    elif sum_supply < sum_demand:
        diff = sum_demand - sum_supply
        costs = np.vstack((costs, np.zeros((1, orig_n))))
        supply = np.append(supply, diff)
        added_row = True
        
    m, n = costs.shape
    
    basis_matrix = np.full((m, n), None, dtype=object)
    s = supply.copy()
    d = demand.copy()
    i, j = 0, 0
    while i < m and j < n:
        val = min(s[i], d[j])
        basis_matrix[i, j] = val
        s[i] -= val
        d[j] -= val
        if s[i] == 0 and i < m - 1 and (d[j] != 0 or j == n - 1):
            i += 1
        elif d[j] == 0 and j < n - 1:
            j += 1
        else:
            i += 1
            j += 1

    iteration = 0
    while True:
        iteration += 1
        title_str = f"Итерация {iteration}"
        display_transportation_table(costs, basis_matrix, supply, demand, title=title_str)
        
        alpha, beta = [None] * m, [None] * n
        alpha[0] = 0.0
        changed = True
        while changed:
            changed = False
            for r in range(m):
                for c in range(n):
                    if basis_matrix[r, c] is not None:
                        if alpha[r] is not None and beta[c] is None:
                            beta[c] = costs[r, c] - alpha[r]
                            changed = True
                        elif beta[c] is not None and alpha[r] is None:
                            alpha[r] = costs[r, c] - beta[c]
                            changed = True
                            
        optimal = True
        min_delta = 0.0
        target_cell = None
        for r in range(m):
            for c in range(n):
                if basis_matrix[r, c] is None:
                    delta = costs[r, c] - (alpha[r] + beta[c])
                    if delta < 0:
                        optimal = False
                        if delta < min_delta:
                            min_delta = delta
                            target_cell = (r, c)
                            
        if optimal:
            break
            
        cycle = find_cycle(basis_matrix, target_cell)
        minus_cells = cycle[1::2]
        theta = min(basis_matrix[r, c] for r, c in minus_cells)
        
        for idx, (r, c) in enumerate(cycle):
            if basis_matrix[r, c] is None: 
                basis_matrix[r, c] = 0.0
            if idx % 2 == 0:
                basis_matrix[r, c] += theta
            else:
                basis_matrix[r, c] -= theta
                
        removed = False
        for r, c in minus_cells:
            if basis_matrix[r, c] == 0 and not removed:
                basis_matrix[r, c] = None
                removed = True

    total_cost = 0
    equation_parts = []
    
    for r in range(m):
        for c in range(n):
            if basis_matrix[r, c] is not None:
                val = basis_matrix[r, c]
                if (added_row and r == m - 1) or (added_col and c == n - 1):
                    continue
                if val > 0:
                    total_cost += val * costs[r, c]
                    equation_parts.append(f"{int(costs[r,c])}·{int(val)}")
                
    equation_str = " + ".join(equation_parts)
    display(HTML("<h4 style='color: #ffffff;'>Оптимальный план найден</h4>"))
    display(Markdown(f"$f = {equation_str} = {int(total_cost)}$"))

def find_cycle(basis, start):
    m, n = basis.shape
    points = [(r, c) for r in range(m) for c in range(n) if basis[r, c] is not None]
    if start not in points: points.append(start)
    path = [start]

    def dfs(curr, move_row):
        nodes = [p for p in points if (p[0] == curr[0] if move_row else p[1] == curr[1]) and p != curr]
        for nxt in nodes:
            if nxt == start and len(path) >= 4 and len(path) % 2 == 0: return True
            if nxt not in path:
                path.append(nxt)
                if dfs(nxt, not move_row): return True
                path.pop()
        return False

    dfs(start, True)
    return path

In [21]:
display(Markdown("#### Сбалансированный"))
c1 = [[2, 3, 4], 
      [1, 2, 5]]
s1 = [20, 40]
d1 = [10, 20, 30]

solve_transportation_universal(c1, s1, d1)

#### Сбалансированный

Пункты,B1,B2,B3,Запасы
A1,210,310,4,20
A2,1,210,530,40
Потребности,10,20,30,60


Пункты,B1,B2,B3,Запасы
A1,210,3,410,20
A2,1,220,520,40
Потребности,10,20,30,60


Пункты,B1,B2,B3,Запасы
A1,2,3,420,20
A2,110,220,510,40
Потребности,10,20,30,60


$f = 4·20 + 1·10 + 2·20 + 5·10 = 180$

In [22]:
display(Markdown("#### С нарушением баланса"))
c2 = [[1, 2, 3], 
      [2, 3, 3]]
s2 = [20, 40]
d2 = [30, 30, 20]

solve_transportation_universal(c2, s2, d2)

#### С нарушением баланса

Пункты,B1,B2,B3,Запасы
A1,120,2,3,20
A2,210,330,3•,40
A3,0,0,020,20
Потребности,30,30,20,80


$f = 1·20 + 2·10 + 3·30 = 130$

## Метод Фогеля

In [45]:
import pandas as pd
import numpy as np
from IPython.display import display

def style_vogel_table_monochrome(df, active_rows, active_cols, n_original, m_original):
    styler = df.style.set_properties(**{
        'text-align': 'center',
        'font-family': 'Arial, sans-serif',
        'padding': '8px',
        'border': '1px solid #333333',
        'background-color': '#1a1a1a',
        'color': '#ffffff'
    })
    
    styler.set_table_styles([
        {
            'selector': 'th', 
            'props': [
                ('background-color', '#2d2d2d'), 
                ('color', '#ffffff'), 
                ('font-weight', 'bold'), 
                ('border', '1px solid #404040')
            ]
        },
        {
            'selector': 'tr:hover', 
            'props': [('background-color', '#222222')]
        }
    ])
    
    styler.set_properties(subset=['a_i'], **{'background-color': '#252525', 'font-weight': 'bold'})
    if 'b_j' in df.index:
        styler.set_properties(subset=pd.IndexSlice[['b_j'], :], **{'background-color': '#252525', 'font-weight': 'bold'})
        
    penalty_cols = [c for c in df.columns if 'Δ' in str(c)]
    penalty_rows = [r for r in df.index if 'Δ' in str(r)]
    
    if penalty_cols:
        styler.set_properties(subset=penalty_cols, **{'background-color': '#1e1e1e', 'font-style': 'italic'})
    if penalty_rows:
        styler.set_properties(subset=pd.IndexSlice[penalty_rows, :], **{'background-color': '#1e1e1e', 'font-style': 'italic'})

    for i in range(n_original):
        for j in range(m_original):
            row_label = f"A{i+1}"
            col_label = f"B{j+1}"
            if i not in active_rows or j not in active_cols:
                if "]" in str(df.loc[row_label, col_label]) and len(str(df.loc[row_label, col_label]).split()) == 1:
                    styler.set_properties(subset=pd.IndexSlice[[row_label], [col_label]], **{'background-color': '#121212', 'color': '#555555'})
                else:
                    styler.set_properties(subset=pd.IndexSlice[[row_label], [col_label]], **{'background-color': '#1c1c1c', 'font-weight': 'bold'})

    return styler

def print_vogel_table_monochrome(costs, supply, demand, allocation, row_penalties_history, col_penalties_history, active_r, active_c):
    n, m = costs.shape
    display_matrix = []
    
    for i in range(n):
        row = []
        for j in range(m):
            tariff = costs[i, j]
            alloc = allocation[i, j]
            if alloc > 0:
                cell_str = f"[{int(tariff)}] {int(alloc)}"
            else:
                cell_str = f"[{int(tariff)}]"
            row.append(cell_str)
        display_matrix.append(row)
        
    df = pd.DataFrame(display_matrix, index=[f"A{i+1}" for i in range(n)], columns=[f"B{j+1}" for j in range(m)])
    df['a_i'] = supply
    
    max_steps = max(len(row_penalties_history), 1)
    for step in range(max_steps):
        col_penalty_vals = []
        for i in range(n):
            if step < len(row_penalties_history) and i in row_penalties_history[step]:
                val = row_penalties_history[step][i]
                col_penalty_vals.append(str(val) if val is not None else "-")
            else:
                col_penalty_vals.append("-")
        df[f"Δ Стр {step+1}"] = col_penalty_vals
        
    demand_row = [f"{int(d)}" for d in demand] + [""] + [""] * max_steps
    df.loc['b_j'] = demand_row
    
    for step in range(len(col_penalties_history)):
        row_penalty_vals = []
        for j in range(m):
            if j in col_penalties_history[step]:
                val = col_penalties_history[step][j]
                row_penalty_vals.append(str(val) if val is not None else "-")
            else:
                row_penalty_vals.append("-")
        row_penalty_vals += [""] * (1 + max_steps)
        df.loc[f"Δ Стл {step+1}"] = row_penalty_vals
        
    beautiful_df = style_vogel_table_monochrome(df, active_r, active_c, n, m)
    display(beautiful_df)

def vogel_approximation_method_monochrome(costs, supply, demand):
    costs = np.array(costs, dtype=float)
    s = np.array(supply, dtype=float)
    d = np.array(demand, dtype=float)
    n, m = costs.shape
    allocation = np.zeros((n, m))
    
    active_rows = list(range(n))
    active_cols = list(range(m))
    row_penalties_history = []
    col_penalties_history = []
    
    iteration = 1
    
    while len(active_rows) > 0 and len(active_cols) > 0:
        if len(active_rows) == 1:
            i = active_rows[0]
            row_penalties_history.append({r: "-" for r in active_rows})
            col_penalties_history.append({c: "-" for c in active_cols})
            for j in sorted(active_cols, key=lambda c_idx: costs[i, c_idx]):
                if d[j] > 0:
                    quantity = min(s[i], d[j])
                    allocation[i, j] = quantity
                    s[i] -= quantity
                    d[j] -= quantity
            break
            
        if len(active_cols) == 1:
            j = active_cols[0]
            row_penalties_history.append({r: "-" for r in active_rows})
            col_penalties_history.append({c: "-" for c in active_cols})
            for i in sorted(active_rows, key=lambda r_idx: costs[r_idx, j]):
                if s[i] > 0:
                    quantity = min(s[i], d[j])
                    allocation[i, j] = quantity
                    s[i] -= quantity
                    d[j] -= quantity
            break

        current_row_penalties = {i: int(sorted([costs[i, c] for c in active_cols])[1] - sorted([costs[i, c] for c in active_cols])[0]) for i in active_rows}
        current_col_penalties = {j: int(sorted([costs[r, j] for r in active_rows])[1] - sorted([costs[r, j] for r in active_rows])[0]) for j in active_cols}
        
        row_penalties_history.append(current_row_penalties)
        col_penalties_history.append(current_col_penalties)
        
        max_row_p = max(current_row_penalties.values())
        max_col_p = max(current_col_penalties.values())
        
        best_i, best_j = -1, -1
        min_cost = float('inf')
        
        if max_row_p >= max_col_p:
            candidate_rows = [i for i, p in current_row_penalties.items() if p == max_row_p]
            for i in candidate_rows:
                for j in active_cols:
                    if costs[i, j] < min_cost:
                        min_cost = costs[i, j]
                        best_i, best_j = i, j
        else:
            candidate_cols = [j for j, p in current_col_penalties.items() if p == max_col_p]
            for j in candidate_cols:
                for i in active_rows:
                    if costs[i, j] < min_cost:
                        min_cost = costs[i, j]
                        best_i, best_j = i, j
                        
        quantity = min(s[best_i], d[best_j])
        allocation[best_i, best_j] = quantity
        s[best_i] -= quantity
        d[best_j] -= quantity
        
        print(f"Итерация {iteration} — Клетка (A{best_i+1}, B{best_j+1}) Объем: {int(quantity)}")
        print_vogel_table_monochrome(costs, supply, demand, allocation, row_penalties_history, col_penalties_history, active_rows, active_cols)
        
        if s[best_i] == 0:
            active_rows.remove(best_i)
        elif d[best_j] == 0:
            active_cols.remove(best_j)
            
        iteration += 1
        
    print_vogel_table_monochrome(costs, supply, demand, allocation, row_penalties_history, col_penalties_history, range(n), range(m))
    return allocation, np.sum(allocation * costs)

costs = np.array([[7, 12, 4, 8, 5], [1, 8, 6, 5, 3], [6, 13, 8, 7, 4]])
supply = [180, 350, 20]
demand = [110, 90, 120, 80, 150]

final_plan, total_cost = vogel_approximation_method_monochrome(costs, supply, demand)
display(Markdown(f"Суммарная стоимость: $f = {int(total_cost)}$"))

Итерация 1 — Клетка (A2, B1) Объем: 110


,B1,B2,B3,B4,B5,a_i,Δ Стр 1
A1,[7],[12],[4],[8],[5],180,1
A2,[1] 110,[8],[6],[5],[3],350,2
A3,[6],[13],[8],[7],[4],20,2
b_j,110,90,120,80,150,,
Δ Стл 1,5,4,2,2,1,,


Итерация 2 — Клетка (A2, B2) Объем: 90


,B1,B2,B3,B4,B5,a_i,Δ Стр 1,Δ Стр 2
A1,[7],[12],[4],[8],[5],180,1,1
A2,[1] 110,[8] 90,[6],[5],[3],350,2,2
A3,[6],[13],[8],[7],[4],20,2,3
b_j,110,90,120,80,150,,,
Δ Стл 1,5,4,2,2,1,,,
Δ Стл 2,-,4,2,2,1,,,


Итерация 3 — Клетка (A3, B5) Объем: 20


...

,B1,B2,B3,B4,B5,a_i,Δ Стр 1,Δ Стр 2,Δ Стр 3,Δ Стр 4
A1,[7],[12],[4],[8],[5],180,1,1,1,1
A2,[1] 110,[8] 90,[6],[5] 80,[3],350,2,2,2,2
A3,[6],[13],[8],[7],[4] 20,20,2,3,3,-
b_j,110,90,120,80,150,,,,,
Δ Стл 1,5,4,2,2,1,,,,,
Δ Стл 2,-,4,2,2,1,,,,,
Δ Стл 3,-,-,2,2,1,,,,,
Δ Стл 4,-,-,2,3,2,,,,,


Итерация 5 — Клетка (A2, B5) Объем: 70


,B1,B2,B3,B4,B5,a_i,Δ Стр 1,Δ Стр 2,Δ Стр 3,Δ Стр 4,Δ Стр 5
A1,[7],[12],[4],[8],[5],180,1,1,1,1,1
A2,[1] 110,[8] 90,[6],[5] 80,[3] 70,350,2,2,2,2,3
A3,[6],[13],[8],[7],[4] 20,20,2,3,3,-,-
b_j,110,90,120,80,150,,,,,,
Δ Стл 1,5,4,2,2,1,,,,,,
Δ Стл 2,-,4,2,2,1,,,,,,
Δ Стл 3,-,-,2,2,1,,,,,,
Δ Стл 4,-,-,2,3,2,,,,,,
Δ Стл 5,-,-,2,-,2,,,,,,


,B1,B2,B3,B4,B5,a_i,Δ Стр 1,Δ Стр 2,Δ Стр 3,Δ Стр 4,Δ Стр 5,Δ Стр 6
A1,[7],[12],[4] 120,[8],[5] 60,180,1,1,1,1,1,-
A2,[1] 110,[8] 90,[6],[5] 80,[3] 70,350,2,2,2,2,3,-
A3,[6],[13],[8],[7],[4] 20,20,2,3,3,-,-,-
b_j,110,90,120,80,150,,,,,,,
Δ Стл 1,5,4,2,2,1,,,,,,,
Δ Стл 2,-,4,2,2,1,,,,,,,
Δ Стл 3,-,-,2,2,1,,,,,,,
Δ Стл 4,-,-,2,3,2,,,,,,,
Δ Стл 5,-,-,2,-,2,,,,,,,
Δ Стл 6,-,-,-,-,-,,,,,,,


Суммарная стоимость: $f = 2300$